### Reports Classification based on LLM Content
* LLM Selected: For databricks-gpt-oss-20b, your workspace reports:
   * Input tokens: 1.0 DBU / 1M tokens
   * Output tokens: 4.286 DBU / 1M tokens
   * Cheapest model available
   * Extremely fast
   * Classification is usually a structured task
   * Works surprisingly well when:
      * categories are well defined
      * prompts are constrained
      * response is JSON only

* Pending Benchark
   * GPT OSS 20B
   * GPT OSS 120B
   * Qwen3.5 122B A10B


In [0]:
PAT_TOKEN = "<TOKEN>"
BASE_URL = "https://adb-196597926638265.5.azuredatabricks.net"

### Select all reports
* All available reports (16k)
* Scheduled and not scheduled, detected in the audited period.
* 5k reports remains, scheduled but not accessed.
* Produce dict using the doc_id as keys containing an array with the detected BU consumers.


In [0]:
v1_sql: str = """
    with investigation_list as (
        with full_list as (
            -- Owner Document IDs
            select distinct cms1.Document_Id, ud1.BU
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
            left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
            on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
            left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
            on upper(trim(ud1.user_ID)) = upper(trim(
                case
                when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
                else cms1.created
                end
            ))
            union all
            -- Viewing Document IDs
            select distinct cms1.Document_Id, ud1.BU
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
            left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
            on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
            left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
            on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
        ),
        inscope_list as (
            select distinct Document_Id
            from full_list
            where Document_Id in (
            select distinct Document_Id
            from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
            )
        ),
        Audit_data as (
            select
            UA1.*,
            case
                when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
                else 'Viewer_events'
            end as usage_category,
            UP1.BU
            from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
            left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
            on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
        ),
        Audit_profile as (
            select
            WEBI_Doc_ID,
            count(distinct UserName) as user_count,
            count(distinct BU) as bu_count
            from Audit_data
            group by WEBI_Doc_ID
        ),
        total_in_scope as (
            select count(distinct Document_Id) as total_cnt
            from (
            select distinct Document_Id
            from full_list
            where Document_Id in (
                select distinct Document_Id
                from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
            )
            )
        )
        select distinct l1.Document_Id as Shortlist_ID
        from inscope_list as l1
        left join Audit_profile as a1
            on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
        where a1.WEBI_Doc_ID is not null
        --   limit 500
        -- This is the position to limit total number of reports extracted
            
        ),
        Audit_data as (
        select
            UA1.*,
            case
            when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
            else 'Viewer_events'
            end as usage_category,
            UP1.BU
        from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
        left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
            on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
        where ua1.WEBI_Doc_ID in (select Shortlist_ID from investigation_list)
        ),
        owner_list as (
        select distinct
            cms1.Document_Id,
            cms1.Document_name,
            cms1.Full_path,
            cms1.scheduled,
            ud2.BU,
            ud2.Is_Controller,
            ud2.Is_Manager
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
        left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
            on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
        left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud2
            on upper(trim(ud2.user_ID)) = upper(trim(
            case
                when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
                else cms1.created
            end
            ))
        where cms1.Document_Id in (select Document_Id from investigation_list)
        )
        select distinct
        investigation_list.Shortlist_ID as Document_ID,
        owner_list.Document_name as Document_name,
        owner_list.Full_path as Folder_path,
        Audit_data.bu as Accessed_BU,
        owner_list.Is_Controller as accessedby_controler,
        owner_list.scheduled,
        owner_list.bu as ownedby_BU,
        owner_list.Is_Manager as ownedby_manager,
        owner_list.Is_Controller as ownedby_controler
        from investigation_list
        left join Audit_data
        on investigation_list.Shortlist_ID = Audit_data.WEBI_Doc_ID
        left join owner_list
        on investigation_list.Shortlist_ID = owner_list.Document_Id
        where Audit_data.UserName is not null
        order by Document_ID, Accessed_BU
"""

v2_sql: str = """
    with investigation_list as (
    select distinct l1.Document_Id as Shortlist_ID
    from (select distinct Document_Id
        from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver) as l1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as a1
        on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID)
    where a1.WEBI_Doc_ID is not null
    --   limit 500
    -- This is the position to limit total number of reports extracted
        
    ),
    Audit_data as (
    select
        UA1.*,
        case
        when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
        else 'Viewer_events'
        end as usage_category,
        up1.Is_Controller,
        UP1.BU,
        up1.BU_Group
    from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
        on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
    where ua1.WEBI_Doc_ID in (select Shortlist_ID from investigation_list)
    ),
    owner_list as (
    select distinct
        cms1.Document_Id,
        cms1.Document_name,
        cms1.Full_path,
        cms1.scheduled,
        cms1.file_location,
        ud2.BU,
        ud2.BU_Group,
        ud2.Is_Controller,
        ud2.Is_Manager
    from (
        select * from (select *,
        row_number() over (partition by Document_Id order by ingestion_ts desc) as rn
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms) as l1
        where l1.rn = 1
        ) as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
        on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud2
        on upper(trim(ud2.user_ID)) = upper(trim(
        case
            when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
            else cms1.created
        end
        ))
    where cms1.Document_Id in (select Document_Id from investigation_list)
    )
    select 
    investigation_list.Shortlist_ID as Document_ID,
    owner_list.Document_name as Document_name,
    owner_list.Full_path as Folder_path,
    owner_list.file_location as CSV_File_location,
    owner_list.scheduled,
    owner_list.bu as ownedby_BU,
    owner_list.BU_Group as Ownedby_BUGroup,
    owner_list.Is_Manager as ownedby_manager,
    owner_list.Is_Controller as ownedby_controler,
    Audit_data.bu as Accessed_BU,
    Audit_data.BU_Group as accessed_BUGroup,
    case 
        when max(Audit_data.Is_Controller) = true then true
        else false
    end as Accessed_by_Controller,
    count(distinct Audit_data.UserName) as Accessed_users
    from investigation_list
    left join Audit_data
    on investigation_list.Shortlist_ID = Audit_data.WEBI_Doc_ID
    left join owner_list
    on investigation_list.Shortlist_ID = owner_list.Document_Id
    where Audit_data.UserName is not null
    group by investigation_list.Shortlist_ID,
    owner_list.Document_name,
    owner_list.Full_path,
    owner_list.file_location,
    owner_list.scheduled,
    owner_list.bu,
    owner_list.BU_Group,
    owner_list.Is_Manager,
    owner_list.Is_Controller,
    Audit_data.bu,
    Audit_data.BU_Group
"""

reports_all = spark.sql(v2_sql)
print(f"Audit traces detected (aggregated by BU): {reports_all.count()}")
reports_all: list = [row.asDict() for row in reports_all.collect()]

# Aggregate consummers by reports
reports_all_grouped: dict = {}
units_detected: list = []
for consumer in reports_all:
    if consumer['Accessed_BU'] not in units_detected:
        units_detected.append(consumer['Accessed_BU'])
    if consumer['ownedby_BU'] not in units_detected:
        units_detected.append(consumer['ownedby_BU'])
    if str(consumer['Document_ID']) not in reports_all_grouped:
        reports_all_grouped[str(consumer['Document_ID'])] = {'consumers_count': 0, 'consumers': []}
    reports_all_grouped[str(consumer['Document_ID'])]['consumers_count'] += 1
    reports_all_grouped[str(consumer['Document_ID'])]['consumers'].append(consumer)

print(f'Unique reports detected in the audit: {len(reports_all_grouped.keys())}')

print(f'Unique units detected: {len(units_detected)}')               

final_report_metadata: dict = {}

for report, metadata in reports_all_grouped.items():
    report_metadata = metadata['consumers'][0].copy()
    report_metadata['Consumers BU Count'] = 0
    report_metadata['Consumers unique users Count'] = 0
    report_metadata['Consumers BU List'] = []
    report_metadata.pop('Accessed_BU')
    report_metadata['Business Unit of the owner'] = report_metadata.pop('ownedby_BU')
    report_metadata['Business Unit Area of the owner'] = report_metadata.pop('Ownedby_BUGroup')
    report_metadata['Report owner is finance or controller'] = report_metadata.pop('ownedby_controler')
    report_metadata['Report owner is manager'] = report_metadata.pop('ownedby_manager')
    report_metadata['Report folder path'] = report_metadata.pop('Folder_path')
    report_metadata['Report is consumed by controllers'] = report_metadata.pop('Accessed_by_Controller')
    for consumer in metadata['consumers']:
        if consumer['Accessed_BU'] is not None:
            report_metadata['Consumers BU List'].append(
                {
                "Business Unit Area": consumer['accessed_BUGroup'],
                "Business Unit": consumer['Accessed_BU'],
                "Unique users": consumer['Accessed_users']
            })
            report_metadata['Consumers BU Count'] += 1
            report_metadata['Consumers unique users Count'] += consumer['Accessed_users']
    if report_metadata['Consumers BU Count'] == 1:
        if report_metadata['Business Unit of the owner'] == report_metadata['Consumers BU List'][0]['Business Unit']:
            report_metadata['Report usage is limited to owner unit'] = True
            if report_metadata['Consumers unique users Count'] == 1:
                report_metadata['Report is for personal usage or investigation'] = True
            else:
                report_metadata['Report is for personal usage or investigation'] = False
    if 'Report usage is limited to owner unit' not in report_metadata:
        report_metadata['Report usage is limited to owner unit'] = False
    if 'Report is for personal usage or investigation' not in report_metadata:
        report_metadata['Report is for personal usage or investigation'] = False
    
    clean_following_keys: list = ['Ownedby_BUGroup', 'accessed_BUGroup', 'Accessed_users']
    for key in clean_following_keys:
        if key in report_metadata:
            report_metadata.pop(key)

    final_report_metadata[report] = report_metadata

display(final_report_metadata)

In [0]:
def print_report(file_path: str) -> None:
    with open(file_path, "r") as report_file:
        print(report_file.read())

def report_scrapper(file_path: str, lines_to_read: int = 5) -> dict:
    with open(file_path, "r") as report_file:
        first_line_flag: bool = False
        reports_found: dict = {}
        report_name: str = None
        report_content: list = []
        find_next_empty_line: bool = False
        for line in report_file: 
            if find_next_empty_line is True:
                if line == '\n':
                    find_next_empty_line: bool = False
                continue
                
            if len(line.split('\t')) > 3 and len(report_content) < lines_to_read:
                report_content.append(line.replace('\t', '|').replace('\n', ''))
            else:
                if len(report_content) == 0:
                    if first_line_flag is False:
                        if len(line) > 5:
                            first_line_flag: bool = True
                            report_name: str = line.replace('\n', '')
                else:
                    reports_found[report_name if report_name is not None else f'Report {len(reports_found.keys())+1}'] = report_content
                    report_content: list = []
                    report_name = None
                    find_next_empty_line: bool = True
                    first_line_flag: bool = False
        
        return reports_found


report_path: str = '/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod/object_type=csv/ingestion_date=2026-06-19/WEBI_18295443.csv'
report_path: str = '/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod/object_type=csv/ingestion_date=2026-06-03/WEBI_46548.csv'
#print_report(file_path=report_path)
reports = report_scrapper(file_path=report_path)

for key, value in reports.items():
    print(f'Report name: {key}')
    for line in value:
        print(line)
    print('-------------------------------------------------------------')




### Get the routes to the reports output (no longer required)
* Ignore reports without extractions
* This process produces an array of dicts.

In [0]:
exports_index = spark.sql("""
    with investigation_list as (
  with full_list as (
    -- Owner Document IDs
    select distinct cms1.Document_Id, ud1.BU
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
      on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
      on upper(trim(ud1.user_ID)) = upper(trim(
        case
          when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
          else cms1.created
        end
      ))
    union all
    -- Viewing Document IDs
    select distinct cms1.Document_Id, ud1.BU
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
      on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
      on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
  ),
  inscope_list as (
    select distinct Document_Id
    from full_list
    where Document_Id in (
      select distinct Document_Id
      from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
    )
  ),
  Audit_data as (
    select
      UA1.*,
      case
        when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
        else 'Viewer_events'
      end as usage_category,
      UP1.BU
    from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
      on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  ),
  Audit_profile as (
    select
      WEBI_Doc_ID,
      count(distinct UserName) as user_count,
      count(distinct BU) as bu_count
    from Audit_data
    group by WEBI_Doc_ID
  ),
  total_in_scope as (
    select count(distinct Document_Id) as total_cnt
    from (
      select distinct Document_Id
      from full_list
      where Document_Id in (
        select distinct Document_Id
        from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
      )
    )
  )
  select distinct l1.Document_Id as Shortlist_ID
  from inscope_list as l1
  left join Audit_profile as a1
    on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
  where a1.WEBI_Doc_ID is not null
--   limit 500
-- This is the position to limit total number of reports extracted
    
),
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
    on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  where ua1.WEBI_Doc_ID in (select Shortlist_ID from investigation_list)
),
owner_list as (
  select distinct
    cms1.Document_Id,
    cms1.Document_name,
    cms1.scheduled,
    ud2.BU,
    ud2.Is_Controller,
    ud2.Is_Manager
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
    on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud2
    on upper(trim(ud2.user_ID)) = upper(trim(
      case
        when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
        else cms1.created
      end
    ))
  where cms1.Document_Id in (select Document_Id from investigation_list)
)
select distinct
  investigation_list.Shortlist_ID as Document_ID,
  cms1.File_location
from investigation_list
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms1
  on trim(investigation_list.Shortlist_ID)=trim(cms1.Document_Id)
where cms1.File_location is not null
""")

exports_index: list = [row.asDict() for row in exports_index.collect()]
exports_index_dict = {}
print(f"Exports available for {len(exports_index)} reports")
for export in exports_index:
    exports_index_dict[str(export['Document_ID'])] = export['File_location'][5:]
display(exports_index_dict)

### Open a random report output to understand its content
* CSV with tab separator
* Comma as decial separator

In [0]:
with open("/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod/object_type=csv/ingestion_date=2026-06-03/WEBI_46548.csv", "r") as file:
    content = file.read().split("\n")
    for line in content:
        print(line)
    

### Get report variables

In [0]:
all_variables = spark.sql("""
    with investigation_list as (
  with full_list as (
    -- Owner Document IDs
    select distinct cms1.Document_Id, ud1.BU
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
      on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
      on upper(trim(ud1.user_ID)) = upper(trim(
        case
          when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
          else cms1.created
        end
      ))
    union all
    -- Viewing Document IDs
    select distinct cms1.Document_Id, ud1.BU
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
      on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
      on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
  ),
  inscope_list as (
    select distinct Document_Id
    from full_list
    where Document_Id in (
      select distinct Document_Id
      from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
    )
  ),
  Audit_data as (
    select
      UA1.*,
      case
        when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
        else 'Viewer_events'
      end as usage_category,
      UP1.BU
    from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
      on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  ),
  Audit_profile as (
    select
      WEBI_Doc_ID,
      count(distinct UserName) as user_count,
      count(distinct BU) as bu_count
    from Audit_data
    group by WEBI_Doc_ID
  ),
  total_in_scope as (
    select count(distinct Document_Id) as total_cnt
    from (
      select distinct Document_Id
      from full_list
      where Document_Id in (
        select distinct Document_Id
        from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver
      )
    )
  )
  select distinct l1.Document_Id as Shortlist_ID
  from inscope_list as l1
  left join Audit_profile as a1
    on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
  where a1.WEBI_Doc_ID is not null
--   limit 500
-- This is the position to limit total number of reports extracted
    
),
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
    on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  where ua1.WEBI_Doc_ID in (select Shortlist_ID from investigation_list)
),
owner_list as (
  select distinct
    cms1.Document_Id,
    cms1.Document_name,
    cms1.scheduled,
    ud2.BU,
    ud2.Is_Controller,
    ud2.Is_Manager
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
    on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud2
    on upper(trim(ud2.user_ID)) = upper(trim(
      case
        when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
        else cms1.created
      end
    ))
  where cms1.Document_Id in (select Document_Id from investigation_list)
)
select distinct
de1.*
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_data_entries as de1
  where trim(de1.Document_Id) in (select Shortlist_ID from investigation_list)
""")

all_variables: list = [row.asDict() for row in all_variables.collect()]

# Aggregate columns by reports
metadata_grouped_by_report: dict = {}
for data_field in all_variables:
    if data_field['Document_Id'] not in metadata_grouped_by_report:
        metadata_grouped_by_report[str(data_field['Document_Id'])] = {'variables_count': 0, 'variables': []}

    new_data_field: dict = {
        "entry_type": data_field['entry_type'],
        "entry_name": data_field['entry_name'],
        "entry_datatype": data_field['entry_datatype'],
        "entry_qualification": data_field['entry_qualification'],
        "extracted_datafield": data_field['extracted_datafield'],
        "Universe_name": data_field['Universe_Name'],
        "datafield_datasourcepath": data_field['datafield_datasourcepath'],
        "datafield_description": data_field['datafield_description'],
        "sql_definition": data_field['sql_definition'],
        "variable_definition": data_field['variable_definition'],
        "datafield_name_universe": data_field['datafield_name_universe'],
    }

    keys = list(new_data_field.keys())
    for key in keys:
        if new_data_field[key] is None:
            new_data_field.pop(key)

    # Skip duplicates
    found: bool = False
    for field_definition in metadata_grouped_by_report[str(data_field['Document_Id'])]['variables']:
        if field_definition == new_data_field:
            found = True
            break
    if not found:
        metadata_grouped_by_report[str(data_field['Document_Id'])]['variables'].append(new_data_field)
        metadata_grouped_by_report[str(data_field['Document_Id'])]['variables_count'] += 1

print(f'Unique reports detected: {len(metadata_grouped_by_report.keys())}')

display(metadata_grouped_by_report[list(metadata_grouped_by_report.keys())[0]])

### Test LLM

In [0]:
# LLM Catalog

import requests
import json

def test_serving_endpoints():
    r = requests.get(
        f"{BASE_URL}/api/2.0/serving-endpoints",
        headers={"Authorization": f"Bearer {PAT_TOKEN}"}
    )

    print(r.status_code)

def test_models_catalog():
    r = requests.get(
        f"{BASE_URL}/api/2.0/serving-endpoints",
        headers={"Authorization": f"Bearer {PAT_TOKEN}"}
    )

    print(json.dumps(r.json(), indent=2))

test_models_catalog()

In [0]:
# COST Calculation for GPT_OSS_20B
GPT_OSS_20B_INPUT_DBU_PER_M = 1.0
GPT_OSS_20B_OUTPUT_DBU_PER_M = 4.286

def store_cost(llm_output: dict) -> dict:
    """
    Store cost for the given model

    Expected usage format:
    {
        "prompt_tokens": 123,
        "completion_tokens": 45,
        "total_tokens": 168
    }
    """
    audit: dict = {
        "model": llm_output['model'],
        "tokens_in": llm_output['usage']['prompt_tokens'],
        "tokens_out": llm_output['usage']['completion_tokens'],
        "prompt_id": llm_output['id']
    }

    with open("02_Tokens_Audit.txt", "a") as f:
        f.write(json.dumps(audit) + "\n")

In [0]:
import requests
import json

url = f"{BASE_URL}/serving-endpoints/databricks-gpt-oss-20b/invocations"
headers = {
    "Authorization": f"Bearer {PAT_TOKEN}",
    "Content-Type": "application/json"
}

data = {
    "messages": [
        {
            "role": "system",
            "content": """
                You are a data transformation engine.
                You are not a chatbot.

                Your task is to enrich Business Intelligence report metadata.

                Return exactly one valid JSON object.

                Do not return markdown, explanations, comments, code fences, reasoning or analysis.

                =================================================
                OBJECTIVE
                =================================================

                Your responsibilities are:

                1. Translate the report title to English.
                2. Generate a business-friendly report description.
                3. Identify monitored business entities.
                4. Classify the report purpose.
                5. Classify the geographical scope.
                6. Classify the primary entity subject.
                7. Estimate confidence.
                8. Produce a standardized JSON output.

                =================================================
                INPUT
                =================================================

                You will receive report metadata.

                Possible metadata fields may include:

                - report title
                - report owner
                - report consumers
                - folder path
                - report description
                - additional metadata

                Use only the information provided.

                Do not use external knowledge.

                =================================================
                EVIDENCE RULES
                =================================================

                Every classification decision must be supported by evidence found in the input metadata.

                Evidence may come from:

                - report title
                - report description
                - folder path
                - additional metadata
                - owner
                - consumers

                Evidence priority:

                1. report title
                2. report description
                3. folder path
                4. additional metadata
                5. owner
                6. consumers

                Owner and consumers are weak signals.

                Do not use owner or consumers as primary evidence when business terminology is available elsewhere.

                If evidence sources conflict, use the highest priority evidence source.

                If evidence is insufficient:

                - generate a generic but accurate description
                - reduce confidence
                - use Unknown when appropriate

                Do not maximize completeness.
                Maximize accuracy.

                =================================================
                DESCRIPTION RULES
                =================================================

                The report description must be grounded exclusively in the provided metadata.

                Only use information that is:

                1. Explicitly present in the metadata.
                2. Directly supported by business terminology appearing in the metadata.

                Every significant statement included in the description must be supported by classification evidence.

                If a statement cannot be traced to evidence, do not include it.

                The description must:

                - be business-friendly
                - be understandable by non-technical users
                - explain the likely purpose of the report
                - mention monitored entities only when supported by evidence
                - contain between 15 and 50 words
                - be a single sentence
                - avoid technical implementation details
                - avoid references to universes, databases, SQL, ETL, Power BI, Databricks or technologies
                - avoid assumptions
                - avoid speculation
                - avoid invented facts

                When insufficient information is available:

                - generate a generic but accurate description
                - prefer accuracy over detail

                Example:

                Good:
                "Provides visibility into invoice types and their evolution over time within the Nordic business area."

                Bad:
                "Provides visibility into invoice amounts, customer billing trends, broker performance and revenue evolution."

                The bad example is invalid unless those concepts are explicitly supported by metadata.

                =================================================
                ENTITY EXTRACTION RULES
                =================================================

                Identify only entities that are explicitly mentioned or strongly supported by metadata.

                Possible entities include:

                - Customer
                - Client
                - Policyholder
                - Buyer
                - Broker
                - Policy
                - Claim
                - Invoice
                - Contract
                - Credit Limit
                - Recovery Case
                - Business Unit
                - Product
                - Cost Centre
                - Declaration
                - Premium
                - Country
                - Reinsurance
                - Portfolio

                Only include entities supported by evidence.

                =================================================
                REPORT PURPOSE CLASSIFICATION
                =================================================

                Select exactly one value:

                - Management
                - Operational
                - Regulatory
                - Analytical
                - Ad-hoc
                - Unknown

                Definitions:

                Management:
                Used to monitor business performance, budgets, profitability or strategic execution.

                Operational:
                Supports day-to-day operations, monitoring, workload management or process execution.

                Regulatory:
                Created primarily for regulatory, compliance or audit requirements.

                Analytical:
                Intended for investigation, trend analysis, segmentation or business insights.

                Ad-hoc:
                Created for a specific temporary request or one-off analysis.

                Unknown:
                Insufficient evidence exists.

                Generate a purpose description of maximum 20 words.

                =================================================
                GEOGRAPHICAL SCOPE CLASSIFICATION
                =================================================

                Select exactly one value:

                - Domestic
                - Cross-Border
                - Unknown

                Definitions:

                Domestic:
                Single country, local market or local business operation.

                Cross-Border:
                Multiple countries, regions, global operations or international activity.

                Unknown:
                Insufficient evidence exists.

                Generate a geo scope description of maximum 20 words.

                =================================================
                ENTITY SUBJECT CLASSIFICATION
                =================================================

                Select exactly one value:

                - Client / Policyholder
                - Buyer
                - Both
                - Not Entity-Specific
                - Unknown

                Definitions:

                Client / Policyholder:
                Focused on customers, insureds, brokers or policyholders.

                Buyer:
                Focused on buyers, buyer risk, buyer exposure or buyer performance.

                Both:
                Focused on relationships between policyholders and buyers.

                Not Entity-Specific:
                Focused on processes, claims, premiums, finance, products, operations or systems.

                Unknown:
                Insufficient evidence exists.

                Generate an entity subject description of maximum 20 words.

                =================================================
                CONFIDENCE RULES
                =================================================

                Use only:

                - low
                - medium
                - high
                - very-high

                Guidance:

                very-high:
                Strong direct evidence.

                high:
                Strong evidence with minor ambiguity.

                medium:
                Limited metadata available.

                low:
                Weak or ambiguous evidence.

                Confidence must reflect the weakest relevant part of the classification.

                =================================================
                FINAL VALIDATION
                =================================================

                Before returning the result verify:

                1. Every monitored entity is supported by evidence.
                2. Every important statement in the description is supported by evidence.
                3. Purpose is valid.
                4. Geo scope is valid.
                5. Entity subject is valid.
                6. Confidence is consistent with evidence.
                7. JSON is valid.
                8. No required field is missing.

                =================================================
                OUTPUT
                =================================================

                Return valid JSON only. Do not use \n in the json structure.

                {
                "report_name_in_english": "",
                "report_description": "",
                "monitored_entities": [],
                "classification_evidence": [],
                "report_purpose": "",
                "report_purpose_description": "",
                "geo_scope": "",
                "geo_scope_description": "",
                "entity_subject": "",
                "entity_subject_description": "",
                "confidence": ""
                }
            """
        },
        {
            "role": "user",
            "content": """
             {
                'Document_ID': 46553,
                'Document_name': 'New GL Journals details by account and start to end dates AU 259 - Fixed Asset',
                'scheduled': False,
                'Consumers Count': 1,
                'Consumers List': ['COMON-Oceania'],
                'Business Unit of the owner': 'COMON-Oceania',
                'Report owner is finance or controller': True,
                'Report owner is manager': False,
                'Report folder path': 'My Folders',
                'Report is consumed by controllers': True,
                'Report usage is limited to owner unit': 'Yes',
                'SAP BO variables': {'variables_count': 24,
                'variables': [{'entry_type': 'DICTIONARY',
                    'entry_name': 'Accounted Net',
                    'entry_datatype': 'Numeric',
                    'entry_qualification': 'Measure',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Lines',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Balance Sheet Rolls',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Company Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Business Type',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Header.Description',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Header',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Underwriting Year',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Je Source',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Header',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Name',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Header',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Project Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Accounted Cr',
                    'entry_datatype': 'Numeric',
                    'entry_qualification': 'Measure',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Lines',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Lines.Description',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Lines',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'InterCompany',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Creation Date',
                    'entry_datatype': 'DateTime',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Header',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'GL Period Name - Obsolete',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\GL Periods',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Account Code Description',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Organisation Unit',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'GL Period Number - Obsolete',
                    'entry_datatype': 'Numeric',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\GL Periods',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Country Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Account Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Currency Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Header',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Reserve Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Analysis Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Product Code',
                    'entry_datatype': 'String',
                    'entry_qualification': 'Dimension',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\FLEXFIELD\\Segments - New',
                    'universe_primary_catagory': 'Finance / Accounting'},
                {'entry_type': 'DICTIONARY',
                    'entry_name': 'Accounted Dr',
                    'entry_datatype': 'Numeric',
                    'entry_qualification': 'Measure',
                    'Universe_name': 'None',
                    'datafield_datasourcepath': '\\GENERAL LEDGER\\Journals\\Lines',
                    'universe_primary_catagory': 'Finance / Accounting'}]},
                'Report sample': 'Report Title\n\nJe Source;Company Code;Account Code;Account Code Description;Lines.Description;Balance Sheet Rolls;Organisation Unit;InterCompany;Analysis Code;Underwriting Year;Project Code;Business Type;Product Code;Country Code;Reserve Code;Creation Date;GL Period Name - Obsolete;Currency Code;GL Period Number - Obsolete;Header.Description;Name;Accounted Dr;Accounted Cr;Accounted Net\nPayables;259;23300;IT Minor Purchases;5x Cable charger for iPhones, sub total 103.95 tax included + delivery cost AUD12.09.Total:116.04;00;5024;000;0000000000;0000;IC0247;00;00;000000;0000;09/06/17;JUN-17;AUD;6;Journal Import 16778456:;08-JUN-2017 Purchase Invoices AUD;189;94.5;94.5\nPayables;259;23300;IT Minor Purchases;BELKIN LIGHTING CABLE 1.2m - RED;00;5024;000;0000000000;0000;IC0247;00;00;000000;0000;09/06/17;JUN-17;AUD;6;Journal Import 16778456:;08-JUN-2017 Purchase Invoices AUD;0;0;0\nPayables;259;23300;IT Minor Purchases;Delivery;00;5024;000;0000000000;0000;IC0247;00;00;000000;0000;09/06/17;JUN-17;AUD;6;Journal Import 16778456:;08-JUN-2017 Purchase Invoices AUD;24.18;13.19;10.99\nPayables;259;23310;Consumables;Print\\Copy pay per click for Australia - 01/01/2017 - 31/12/2017;00;5024;000;0000000000;0000;IC0244;00;00;000000;0000;27/06/17;JUN-17;AUD;6;Journal Import 16938851:;26-JUN-2017 Purchase Invoices AUD;646.61;0;646.61\nPayables;259;23310;Consumables;RENTAL CHARGE 01/06/17 - 30/06/17;00;5024;000;0000000000;0000;IC0244;00;00;000000;0000;27/06/17;JUN-17;AUD;6;Journal Import 16938851:;26-JUN-2017 Purchase Invoices AUD;64.66;64.66;0\nPayables;259;24000;Consultancy;PAYROLL PROCESSING  12/05/17;00;6002;000;0000000000;0000;000000;00;00;000000;0000;09/06/17;JUN-17;AUD;6;Journal Import 16778456:;08-JUN-2017 Purchase Invoices AUD;1,987.99;0;1,987.99\nPayables;259;24010;Legal assistance;Canegrowers Brisbane Lease - Assignment Atradius to Compania etc - ATR/509/4 [EMC-DOCS.FID80443];00;6201;000;0000000000;0000;401078;00;00;000000;0000;23/06/17;JUN-17;AUD;6;Journal Import 16903105:;22-JUN-2017 Purchase Invoices AUD;500;0;500\nPayables;259;24010;Legal assistance;LEGAL FEES - PAPER FORCE (OCEANIA) PL - REVIEW OF INSURANCE;00;1101;000;0000000000;0000;700502;00;00;000000;0000;23/06/17;JUN-17;AUD;6;Journal Import 16903105:;22-JUN-2017 Purchase Invoices AUD;1,363.5;0;1,363.5\nPayables;259;24010;Legal assistance;PROFESSIONAL FEES- POTENTIAL REORGANISATION- PROJECT MINERVA - PERIOD 19/04/2017- 12/05/2017;00;6201;000;0000000000;0000;401078;00;00;000000;0000;27/06/17;JUN-17;AUD;6;Journal Import 16938851:;26-JUN-2017 Purchase Invoices AUD;1,560;0;1,560\nPayables;259;25000;Telephone;RECURRING CHARGES;00;5025;000;0000000000;0000;000000;00;00;000000;0000;07/06/17;JUN-17;AUD;6;Journal Import 16761714:;07-JUN-2017 Purchase Invoices AUD;700;0;700\nPayables;259;25100;Postage;COURIER FROM SYD- SINGAPORE + SYD-INDONESIA - IT;00;5024;000;0000000000;0000;000000;00;00;000000;0000;14/06/17;JUN-17;AUD;6;Journal Import 16816618:;13-JUN-2017 Purchase Invoices AUD;113.26;0;113.26\nPayables;259;26000;Internal Travel Transportation;ACCOMMODATION FOR BEAN OUYANG 04/06/17 TO 01/07/17;00;4011;000;0000000000;0000;000000;00;00;000000;0000;07/06/17;JUN-17;AUD;6;Journal Import 16761714:;06-JUN-2017 Purchase Invoices AUD;4,652.84;4,652.84;0\n'}
            """
        }
    ],
    "max_tokens": 1500,
    "temperature": 0
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:    
    print(json.dumps(response.json(), indent=2))
    store_cost(response.json())
else:
    print(f"Error: {response.status_code}")
    print(response.json())

### Execution logic
* High Level
    * Driven by reports metadata
    * Enrich with report variables
    * Enrich with report sample
* Process
    * Final JSON produced one by one (just on time)
    * Verify that the Json has not been processed (processed_reports_registry.txt)
    * Process it
    * Store the result in a file (processed_reports_result.txt)

In [0]:
import csv
import json

def read_lines_with_semicolon(file_path: str, samples: int = 15) -> list:
    lines = []
    text = ""
    with open(file_path, 'r', encoding='utf-8') as file:
        for i, line in enumerate(file):
            if i >= samples:
                break
            lines.append(line.replace('\t', ';').rstrip('\n'))

    for line in lines:
        text += line + '\n'
    return text

def report_output_parser(file_path: str, samples: int = 15) -> object:
        print(file_path)
        sample = {}
        with open(file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file, delimiter='\t')
            rows = []
            for i, row in enumerate(reader):
                if i >= samples:
                    break
                rows.append(row)
            if rows:
                for header in rows[0].keys():
                    sample[header] = [row[header] for row in rows]
        
        if len(sample.keys()) == 1:
            sample = read_lines_with_semicolon(file_path, samples)
        
        return sample


visualize_row: int = 3
row_count: int = 0
for report_id, report_metadata in final_report_metadata.items():
    if report_id in metadata_grouped_by_report:
        if report_id in exports_index_dict:
            report_metadata_enriched: dict = report_metadata.copy()
            report_metadata_enriched['SAP BO variables'] = metadata_grouped_by_report[report_id]
            report_metadata_enriched['Report sample'] = read_lines_with_semicolon(exports_index_dict[report_id])
            row_count += 1
            if row_count == visualize_row:
                display(json.loads(json.dumps(report_metadata_enriched, indent=2)))
                break
        

In [0]:
Prompt_logic="""
You are an Atradius Business Capability Classification Engine.

You are not a chatbot.

Your task is to classify a Business Intelligence report into exactly one Atradius business domain.

Return valid JSON only.

Do not return markdown.
Do not return explanations outside the JSON.

=================================================
OBJECTIVE
=================================================

Analyze the report metadata and determine which Atradius business domain the report most strongly belongs to.

Use only the metadata provided.

Do not use external knowledge.

=================================================
ATRADIUS DOMAINS
=================================================

Select exactly one domain:

- Strategy and Compliance Management
- Market Management
- Corporate Relations and Services
- Distribution & Channel Management
- Customer Relationship Management
- Contract Management
- Buyer Underwriting
- Claims and Recovery Management
- Collections Management
- Financial Compliance Management
- HR Management
- IT Management
- Facility Management
- Procurement Management
- Product Management
- Change Management
- Process Management
- Document Management
- Output Management
- Data Management
- Unknown

=================================================
DOMAIN GUIDANCE
=================================================

Financial Compliance Management:
Accounting, ledger, finance, journals, financial reporting, treasury, provisioning, audits, compliance.

Customer Relationship Management:
Customer portfolios, customer activity, customer onboarding, customer interactions, sales funnels.

Contract Management:
Policies, contracts, invoices, policy issuance, renewals.

Buyer Underwriting:
Buyers, credit limits, buyer risk, underwriting.

Claims and Recovery Management:
Claims, recoveries, disputes, collections after claim events.

Collections Management:
Debt collection operations, collection registration, collection activities.

Distribution & Channel Management:
Brokers, agencies, partners, channels, portals, touchpoints.

Market Management:
Market analysis, segmentation, campaigns.

Corporate Relations and Services:
Legal matters, communications.

Product Management:
Products, product analysis, product development.

IT Management:
Applications, systems, IT operations, infrastructure.

Document Management:
Documents, archives, document workflows.

Data Management:
Data quality, master data, metadata, data governance.

For all other domains, use the strongest available evidence.

=================================================
EVIDENCE RULES
=================================================

Evidence priority:

1. Report title
2. Report description
3. Folder path
4. Business objects variables
5. Additional metadata
6. Owner information
7. Consumer information

Owner and consumer information are weak signals.

If evidence conflicts, prefer the highest priority source.

=================================================
CONFIDENCE
=================================================

Use only:

- low
- medium
- high
- very-high

=================================================
OUTPUT
=================================================

Return valid JSON only:

{
  "atradius_domain": "",
  "confidence": "",
  "classification_evidence": [],
  "classification_reason": ""
}
"""